## Overview

When optimizing models, you need to know **which layers** are the bottlenecks. This tutorial covers advanced profiling techniques in `fasterbench` for identifying performance issues:

| Tool | Use Case |
|------|----------|
| **LayerProfiler** | Unified per-layer analysis (speed, memory, size, compute) |
| **sweep_batch_sizes()** | Find optimal batch size for throughput |
| **sweep_threads()** | Find optimal CPU thread count |
| **sweep_latency()** | Analyze latency vs input resolution |

## 1. LayerProfiler: Unified Per-Layer Analysis

The `LayerProfiler` class provides a unified interface for profiling multiple metrics per layer. This is the recommended approach for comprehensive layer-level analysis.

In [ ]:
import torch
import pandas as pd
from torchvision.models import resnet18
from fasterbench.profiling import LayerProfiler

model = resnet18()
dummy = torch.randn(1, 3, 224, 224)

# Create profiler
profiler = LayerProfiler(model, dummy)

# Profile multiple metrics at once
results = profiler.profile(["speed", "size", "memory"], device="cpu", warmup=3, steps=10)

### Available Metrics

| Metric | Columns Added | Description |
|--------|---------------|-------------|
| `speed` | `speed_ms`, `speed_percent` | Forward pass latency per layer |
| `memory` | `memory_mib`, `memory_percent` | Output tensor size (activation memory) |
| `size` | `params`, `params_percent` | Parameter count per layer |
| `compute` | `macs`, `macs_percent` | MACs per layer (requires torchprofile) |

### Utility Methods: top() and summary()

The `LayerProfiler` provides convenient methods to quickly identify bottlenecks after profiling:

In [ ]:
# Get top 5 slowest layers
print("Top 5 slowest layers:")
for r in profiler.top("speed", n=5):
    print(f"  {r['name']:30} {r['speed_ms']:.3f} ms ({r['speed_percent']:.1f}%)")

# Get top 5 fastest layers (ascending order)
print("\nTop 5 fastest layers:")
for r in profiler.top("speed", n=5, ascending=True):
    print(f"  {r['name']:30} {r['speed_ms']:.3f} ms")

# Get layers with most parameters
print("\nTop 5 layers by parameter count:")
for r in profiler.top("size", n=5):
    print(f"  {r['name']:30} {r['params']:>12,} params")

Top 5 slowest layers:
  layer3.0.downsample.0          55.668 ms (4.2%)
  layer4.1.conv1                 52.793 ms (4.0%)
  layer3.0.conv2                 52.228 ms (4.0%)
  layer4.0.downsample.0          51.313 ms (3.9%)
  layer3.1.conv1                 50.951 ms (3.9%)

Top 5 fastest layers:
  layer4.1.relu                  0.024 ms
  layer4.0.relu                  0.024 ms
  avgpool                        0.085 ms
  layer1.1.bn2                   9.868 ms
  layer1.1.bn1                   10.786 ms

Top 5 layers by parameter count:
  layer4.1.conv1                  2,359,296.0 params
  layer4.0.conv2                  2,359,296.0 params
  layer4.1.conv2                  2,359,296.0 params
  layer4.0.conv1                  1,179,648.0 params
  layer3.0.conv2                    589,824.0 params


In [ ]:
# Print formatted summary of all profiled metrics
profiler.summary(top=10)

═══ Speed (slowest) ═══════════════════════════════════
  layer3.0.downsample.0                    Conv2d            55.668 ms (  4.2%)
  layer4.1.conv1                           Conv2d            52.793 ms (  4.0%)
  layer3.0.conv2                           Conv2d            52.228 ms (  4.0%)
  layer4.0.downsample.0                    Conv2d            51.313 ms (  3.9%)
  layer3.1.conv1                           Conv2d            50.951 ms (  3.9%)
  layer1.1.conv2                           Conv2d            50.742 ms (  3.9%)
  layer3.1.conv2                           Conv2d            49.832 ms (  3.8%)
  layer1.0.conv1                           Conv2d            49.721 ms (  3.8%)
  layer1.0.conv2                           Conv2d            48.559 ms (  3.7%)
  layer4.0.conv2                           Conv2d            48.338 ms (  3.7%)

═══ Parameters (largest) ══════════════════════════════
  layer4.1.conv1                           Conv2d             2,359,296 ( 20.2%)
  laye

## 2. Batch Size Sweeping

Find the optimal batch size for maximum throughput. Larger batches improve GPU utilization but eventually hit memory limits.

In [ ]:
from fasterbench import sweep_batch_sizes

if torch.cuda.is_available():
    results = sweep_batch_sizes(
        model,
        input_shape=(3, 224, 224),  # Shape WITHOUT batch dimension
        batch_sizes=[1, 2, 4, 8, 16, 32],
        device="cuda",
        warmup=10,
        steps=50
    )
    
    print("Batch Size Analysis:")
    print("-" * 60)
    print(f"{'Batch':>6} {'Latency':>10} {'Per-Sample':>12} {'Throughput':>12}")
    print(f"{'Size':>6} {'(ms)':>10} {'(ms)':>12} {'(inf/s)':>12}")
    print("-" * 60)
    for r in results:
        if 'throughput_s' in r and not pd.isna(r.get('mean_ms')):
            print(f"{r['batch_size']:>6} {r['mean_ms']:>10.2f} {r['latency_per_sample_ms']:>12.3f} {r['throughput_s']:>12.1f}")

Batch Size Analysis:
------------------------------------------------------------
 Batch    Latency   Per-Sample   Throughput
  Size       (ms)         (ms)      (inf/s)
------------------------------------------------------------
     1       1.01        1.014        986.3
     2       0.94        0.471       2121.2
     4       1.01        0.253       3951.5
     8       2.48        0.310       3224.2
    16       2.46        0.154       6494.8
    32       2.77        0.087      11538.1


## 3. Thread Count Sweeping (CPU)

For CPU inference, the number of threads significantly impacts performance. More threads isn't always better due to contention.

In [ ]:
from fasterbench import sweep_threads
import os

num_cores = os.cpu_count()
thread_counts = [t for t in [1, 2, 4, 8, 16, 32] if t <= num_cores]

results = sweep_threads(model, dummy, thread_counts=thread_counts, warmup=10, steps=30)

print("Thread Count Analysis:")
print("-" * 50)
print(f"{'Threads':>8} {'Latency (ms)':>15} {'Throughput':>15}")
print("-" * 50)
for r in results:
    print(f"{r['threads']:>8} {r['mean_ms']:>15.2f} {r['throughput_s']:>15.1f}")

Thread Count Analysis:
--------------------------------------------------
 Threads    Latency (ms)      Throughput
--------------------------------------------------
       1           83.75            11.9
       2           89.51            11.2
       4           96.31            10.4
       8           93.26            10.7
      16           49.31            20.3
      32           49.27            20.3


## 4. Input Resolution Sweeping

For vision models, latency scales with input resolution. Use this to find the right speed/accuracy trade-off:

In [ ]:
from fasterbench import sweep_latency

shapes = [
    (1, 3, 128, 128),
    (1, 3, 224, 224),
    (1, 3, 384, 384),
    (1, 3, 512, 512),
]

results = sweep_latency(model, shapes, device="cpu", warmup=5, steps=20)

print("Resolution Analysis:")
print("-" * 50)
print(f"{'Shape':>20} {'Latency (ms)':>15} {'Throughput':>12}")
print("-" * 50)
for r in results:
    print(f"{r['shape']:>20} {r['mean_ms']:>15.2f} {r['throughput_s']:>12.1f}")

Resolution Analysis:
--------------------------------------------------
               Shape    Latency (ms)   Throughput
--------------------------------------------------
         1×3×128×128           16.79         59.6
         1×3×224×224         1001.36          1.0
         1×3×384×384         4375.58          0.2
         1×3×512×512         1731.20          0.6


## Summary

| Tool | Use Case |
|------|----------|
| `LayerProfiler` | Comprehensive per-layer analysis (speed, memory, size, compute) |
| `LayerProfiler.top()` | Get top N layers sorted by any metric |
| `LayerProfiler.summary()` | Print formatted summary of all metrics |
| `sweep_batch_sizes()` | Find optimal batch size for throughput |
| `sweep_threads()` | Find optimal CPU thread count |
| `sweep_latency()` | Analyze latency vs input resolution |

---

## See Also

- [Getting Started Tutorial](benchmark.html) - Basic benchmarking with `benchmark()`
- [Sensitivity Analysis](sensitivity.html) - Analyze layer importance for pruning
- [Profiling API](../analysis/profiling.html) - Full LayerProfiler reference
- [Speed Metrics](../metrics/speed.html) - Detailed speed measurement options